# ELIZA i LLM — od reguł do złudzenia rozumienia

**Podstawy AI · Laboratorium 1 (studia niestacjonarne)**

Dzisiejsze zajęcia mają dwie części z przerwą pośrodku:

| Część | Co robisz |
|---|---|
| **Część I — ELIZA** | |
| 1. Trochę historii | czytasz (5 min) |
| 2. Wyrażenia regularne | uruchamiasz przykłady |
| 3. Budowa ELIZY | Zadania 1–2: dopisujesz zaimki i reguły, testujesz |
| 4. Złam ELIZĘ | Zadanie 3 |
| ☕ **Przerwa** | |
| **Część II — ELIZA kontra LLM** | |
| 5. Połączenie z modelem przez API | klucz, klient, test połączenia |
| 6. Ta sama rozmowa, dwa silniki | Zadanie 4 |
| 7. Łamiące zdania — druga runda | Zadanie 5 |
| 8. Pamięć to lista wiadomości | Zadanie 6 |
| 9. Rola to tekst | Zadanie 7 |
| 10. Refleksja | Zadanie 8 |

Kod ELIZY jest **prawie gotowy** — Twoim zadaniem jest go uzupełnić, uruchomić,
przetestować i zrozumieć, *dlaczego* działa (albo nie działa).

**Potrzebujesz:** klucza Gemini API do części II (instrukcja na slajdach / mailu).

> Teza, którą sprawdzamy przez całe zajęcia: **wrażenie rozumienia powstaje
> po stronie człowieka, nie maszyny.**

# Część I — ELIZA

## 1. Trochę historii

**Kto i kiedy.** ELIZĘ napisał **Joseph Weizenbaum** w MIT w latach 1964–1966.
Opis ukazał się w styczniu 1966 r. w *Communications of the ACM*. Program działał
na komputerze IBM 7094 — użytkownik pisał na dalekopisie, a odpowiedź
drukowała się po chwili.

**Skąd nazwa.** Od Elizy Doolittle z *Pigmaliona* G. B. Shawa — kwiaciarki,
którą uczy się „mówić jak dama". Program też można było „uczyć" mówić,
podmieniając mu zestaw reguł (tzw. *skrypt*).

**DOCTOR.** Najsłynniejszy skrypt udawał psychoterapeutę w nurcie
**rogeriańskim** (Carl Rogers). Taki terapeuta niewiele mówi od siebie —
głównie odbija i parafrazuje słowa pacjenta: *„Mówisz, że…?"*,
*„Jak się z tym czujesz?"*. W tej roli milczenie o świecie wygląda na
profesjonalizm, a nie na niewiedzę. Program nie musiał nic wiedzieć —
wystarczyło przekształcać zdania.

**Efekt ELIZY.** Ludzie bardzo szybko przypisywali programowi zrozumienie
i empatię. Sekretarka Weizenbauma — która widziała, jak program powstaje —
po kilku wymianach zdań poprosiła go, by wyszedł z pokoju. Tę skłonność
nazywamy dziś **efektem ELIZY**.

**Reakcja twórcy.** Weizenbauma to zaniepokoiło. W książce
*Computer Power and Human Reason* (1976) krytykował pomysł, by maszynom
powierzać decyzje wymagające ludzkiego osądu i troski.

**Jak działa** — trzy operacje, nic więcej:

1. **słowo kluczowe** — znajdź w zdaniu wyzwalacz,
2. **dekompozycja** — dopasuj wzorzec z „dziurą", np. `potrzebuję (.*)`,
3. **rekompozycja** — wstaw wycięty fragment do szablonu, odwracając zaimki.

Nie ma tu pamięci, uczenia ani reprezentacji znaczenia.

## 2. Wyrażenia regularne — tyle, ile potrzeba

**Wyrażenie regularne** (*regex*) to wzorzec opisujący, jak wygląda fragment
tekstu. W Pythonie obsługuje je moduł `re`. Wzorce zapisujemy jako surowe
napisy `r"..."`.

| Wzorzec | Znaczenie | Przykład |
|---|---|---|
| `.` | dowolny jeden znak | `k.t` → kot, kit |
| `*` | poprzedni element 0 lub więcej razy | `ha*` → h, ha, haaa |
| `.*` | „cokolwiek" — nasza **dziura** | |
| `( )` | **grupa** — zapamiętaj, co pasowało | `potrzebuję (.*)` |
| `a\|b` | a **albo** b | `matka\|ojciec` |
| `\b` | granica słowa | `\bkot\b` nie trafi w „kotlet" |
| `[^\w\s]` | znak, który nie jest literą ani spacją (interpunkcja) | |

| Funkcja | Co robi |
|---|---|
| `re.search(wzorzec, tekst)` | szuka wzorca gdziekolwiek w tekście; zwraca dopasowanie albo `None` |
| `m.group(1)` / `m.groups()` | fragment złapany przez grupę / krotka wszystkich grup |
| `re.sub(wzorzec, zamiennik, tekst)` | zamienia wystąpienia wzorca |

Uruchom komórki i przyjrzyj się wynikom.

In [10]:
import re
import random

m = re.search(r"potrzebuję (.*)", "chyba potrzebuję urlopu")
print(m.group(0))   # całe dopasowanie
print(m.group(1))   # to, co złapała grupa (.*)
print(m.groups())   # krotka wszystkich grup

print(re.search(r"potrzebuję (.*)", "wszystko w porządku"))   # brak dopasowania -> None

potrzebuję urlopu
urlopu
('urlopu',)
None


In [17]:
# Dwie pułapki: przecinek po słowie kluczowym i kropka na końcu
print(re.search(r"potrzebuję (.*)", "potrzebuję, żeby ktoś mnie słuchał"))   # None!
print(re.search(r"czuję się (.*)", "czuję się samotny.").group(1))            # z kropką

# Rozwiązanie: NORMALIZUJEMY tekst — małe litery, bez interpunkcji
def normalizuj(tekst):
    return re.sub(r"[^\w\s]", "", tekst.lower()).strip()

print(normalizuj("Potrzebuję, żeby ktoś mnie słuchał!"))

None
samotny.
potrzebuję żeby ktoś mnie słuchał


In [18]:
# Wypełnianie szablonu: .format(*grupy) wstawia grupy w miejsca {0}, {1}, ...
szablon = "Dlaczego chciałbyś {0}?"
grupy = ["mieć więcej czasu"]
print(szablon.format(*grupy))   # poprawnie
print(szablon.format(grupy))    # bez * — cała lista trafia do {0}

Dlaczego chciałbyś mieć więcej czasu?
Dlaczego chciałbyś ['mieć więcej czasu']?


## 3. Budowa ELIZY

Poniżej jest **cała ELIZA**. Brakuje tylko „wiedzy": słownika zaimków
i zestawu reguł. Najpierw przeczytaj kod funkcji i upewnij się, że rozumiesz
każdą linijkę — o to mogę zapytać na zajęciach.

### Mechanizm 1 — odbicie zaimków

ELIZA odwraca perspektywę: *„moja rodzina mnie słucha"* → *„twoja rodzina
ciebie słucha"*. Działa to **w obie strony**: gdy pacjent mówi o terapeucie
(*„o co **ci** chodzi"*), terapeuta odpowiada o sobie (*„o co **mi** chodzi"*).

Funkcja `odbij` jest gotowa. `odbicia.get(s, s)` zwraca odbicie słowa,
a jeśli słowa nie ma w słowniku — samo słowo bez zmian.

### 🔧 Zadanie 1
Uzupełnij słownik `odbicia` — dodaj **co najmniej 6** par, w tym
**co najmniej 2 w kierunku odwrotnym** (od terapeuty do pacjenta, np. `ci → mi`).
Uruchom test pod spodem.

In [11]:
odbicia = {
    # pacjent -> terapeuta
    "jestem": "jesteś",
    "moja": "twoja",
    "mnie": "ciebie",
    "mój": "twój",
    "moje": "twoje",
    "mam": "masz",
    "mi": "ci",
    "ja": "ty",
    # terapeuta -> pacjent (kierunek odwrotny)
    "ci": "mi",
    "tobie": "mnie",
    "twoja": "moja",
    "twój": "mój",
    "twoje": "moje",
    "jesteś": "jestem",
    "masz": "mam",
    "ciebie": "mnie",
    # TODO (Zadanie 1): dopisz min. 6 par, np. mój/twój, moje/twoje, mam/masz, mi/ci, ja/ty
    # oraz min. 2 w kierunku odwrotnym, np. "ci": "mi", "tobie": "mnie"
}

def odbij(fragment):
    slowa = fragment.lower().split()
    return " ".join(odbicia.get(s, s) for s in slowa)

In [16]:
# Test Zadania 1 — powinno przejść bez błędu:
assert odbij("moja rodzina") == "twoja rodzina"
assert odbij("o co ci chodzi") == "o co mi chodzi"   # kierunek odwrotny!
print(odbij("Wydaje ci się że to normalne?"))
print("Zadanie 1: OK")

wydaje mi się że to normalne?
Zadanie 1: OK


### Mechanizmy 2 + 3 — reguły i odpowiedź

Każda reguła to para: **wzorzec** (regex) + **lista szablonów** odpowiedzi.
Funkcja `odpowiedz` (gotowa) sprawdza reguły **po kolei** — wygrywa
**pierwsza** pasująca. Z niej bierze grupy, odbija w nich zaimki i wstawia
do losowo wybranego szablonu.

Ostatnia reguła `r".*"` to **reguła awaryjna** — pasuje do każdego tekstu,
dzięki czemu ELIZA zawsze coś odpowiada.

### 🔧 Zadanie 2
Dopisz **co najmniej 4** własne reguły **przed** regułą awaryjną.
- tekst jest znormalizowany, więc wzorce piszemy **małymi literami, bez interpunkcji**,
- sprawdź, czy szablony są gramatyczne po wstawieniu fragmentu,
- odpowiedz w komentarzu: **dlaczego reguła awaryjna musi być na końcu?**

In [12]:
# (wzorzec, [szablony]) — dopasowujemy na tekście po normalizuj()
reguly = [
    (r"potrzebuję (.*)",
        ["Dlaczego potrzebujesz {0}?", "Czy naprawdę pomogłoby ci {0}?"]),
    (r"czuję się (.*)",
        ["Od jak dawna czujesz się {0}?", "Czy często tak się czujesz?"]),
    (r"chciałbym (.*)",
        ["Dlaczego chciałbyś {0}?", "Co by ci to przyniosło?"]),
    (r"chciałabym (.*)",
        ["Dlaczego chciałabyś {0}?", "Co by ci to przyniosło?"]),
    (r"miałem zrobić (.*)",
        ["Po co miałeś zrobić {0}?", "Kto ci to polecił?"]),
    (r"muszę (.*)",
        ["dlaczego musisz {0}?", "Czy naprawdę musisz?"]),
    (r"kupiłem (.*)",
        ["Dlaczego kupiłeś {0}?", "Jesteś zadowolony z zakupu?"]),
    # TODO (Zadanie 2): dodaj min. 4 własne reguły, np. „boję się (.*)", „chciałbym (.*)",
    #                   „jestem (.*)", „nie wiem", „wszyscy|nikt|zawsze|nigdy" ...
    (r"\b(matka|ojciec|rodzina|siostra|brat)\b",
        ["Opowiedz mi więcej o swojej rodzinie.", "Jak układają się te relacje?"]),
    (r".*",   # reguła awaryjna — ZAWSZE na końcu
        ["Rozumiem. Mów dalej.", "Co masz na myśli?", "Jak się z tym czujesz?"]),
]
# Dlaczego reguła awaryjna musi być na końcu? -> ...
#poniewaz pobiera dowolne zdanie, na zasadzie if else albo default w case, sprawdzamy wszystkie przypadki przypadki ktore przyszly nam na mysl, a potem jak nic nie pasuje to wypisujemy cos co pasuje do jak najwiekszej grupy przypadkow


def odpowiedz(wypowiedz):
    w = normalizuj(wypowiedz)
    for wzorzec, szablony in reguly:
        m = re.search(wzorzec, w)
        if m:
            grupy = [odbij(g) for g in m.groups()]
            return random.choice(szablony).format(*grupy)

In [19]:
TESTY = [
    "Potrzebuję, żeby moja rodzina mnie słuchała.",
    "Czuję się dzisiaj samotny.",
    "Mój ojciec ciągle mnie krytykuje.",
    "Nie wiem, o co ci chodzi.",
    "Wszyscy mnie ignorują.",
    "Boję się, że mnie zostawią.",
]

for zdanie in TESTY:
    print(f"Pacjent: {zdanie}\nELIZA:   {odpowiedz(zdanie)}\n")

Pacjent: Potrzebuję, żeby moja rodzina mnie słuchała.
ELIZA:   Czy naprawdę pomogłoby ci żeby twoja rodzina ciebie słuchała?

Pacjent: Czuję się dzisiaj samotny.
ELIZA:   Od jak dawna czujesz się dzisiaj samotny?

Pacjent: Mój ojciec ciągle mnie krytykuje.
ELIZA:   Opowiedz mi więcej o swojej rodzinie.

Pacjent: Nie wiem, o co ci chodzi.
ELIZA:   Rozumiem. Mów dalej.

Pacjent: Wszyscy mnie ignorują.
ELIZA:   Co masz na myśli?

Pacjent: Boję się, że mnie zostawią.
ELIZA:   Rozumiem. Mów dalej.



In [20]:
# (Opcjonalnie) Rozmowa na żywo. Wpisz 'koniec', by wyjść.
# UWAGA: przy „Run All" ta komórka zatrzyma notebook — uruchamiaj ją ręcznie.
while True:
    tekst = input("Ty: ")
    if tekst.strip().lower() == "koniec":
        break
    print("ELIZA:", odpowiedz(tekst))

KeyboardInterrupt: Interrupted by user

## 4. Złam ELIZĘ

Teraz pracujesz jak adwersarz: **celowo** szukasz wypowiedzi, które obnażają
brak rozumienia. Te zdania przydadzą się po przerwie — puścimy je przez LLM.

### 🔧 Zadanie 3 (kod + odpowiedź pisemna)
Znajdź **3–4 różne typy** sytuacji, w których iluzja pęka. Dla każdej podaj
zdanie, pokaż odpowiedź ELIZY i wyjaśnij, **dlaczego** mechanizm tu zawodzi
(odwołaj się do konkretnej linijki kodu).

Gdzie szukać: negacja · odwołanie do czegoś sprzed kilku zdań (brak pamięci) ·
ironia, żart · niegramatyczne odbicie zaimków · fragment niepasujący do szablonu.

In [ ]:
przyklady_lamiace = [
    "Pożyczysz mi 100 zł?",
    "Dlaczego nie chciałeś pożyczyć mi swojego kabla?",
    "Czy zgodnie z tym co mówiłeś mi wcześniej, skończyłeś już zbierać pomidory ze swojego ogródka?",
    "Za ile chciałbyś sprzedać mi swój dom?",


    # TODO (Zadanie 3): wstaw tu swoje zdania łamiące
]
for zdanie in przyklady_lamiace:
    print(f"> {zdanie}\n  ELIZA: {odpowiedz(zdanie)}\n")

**Twoja analiza (Zadanie 3):**

1. Typ: Prośba o działanie którego nie jest w stanie wykonać — przykład: pożyczka pieniędzy — dlaczego zawodzi: Nie może wyciągnąć klucza i poprawnie dopasować do pytania
2. Typ: Sprawdzanie pamięci o zdarzeniu i prośba o niemożliwą do spełnienia czynność Przykład: Pytanie o rozmowę z przeszłości - dlaczego zawodzi: Nie pamięta rozmowy z przeszłośći, nie może się odnieść, nie ma klucza
3. Typ: Zastosowanie zwrotu w formie dokonanym

---

# ☕ Przerwa

Przed powrotem przygotuj **klucz Gemini API** (aistudio.google.com → *Get API key*).

---

# Część II — ELIZA kontra LLM

Te same zdania trafią teraz do dużego modelu językowego (LLM).
Pytanie na tę część: **które pęknięcia ELIZY LLM naprawia, a które zostają?**

> ⚠️ **Nie wpisuj prawdziwych, osobistych zwierzeń.** W darmowym planie
> dostawca może wykorzystywać przesłane treści do ulepszania usług.
> Rozmawiamy z „terapeutą", ale wszystkie zdania mają być **wymyślone**.

## 5. Połączenie z modelem przez API

**Czym jest LLM.** Duży model językowy to sieć neuronowa wytrenowana na
ogromnych zbiorach tekstu do przewidywania, jaki fragment tekstu (*token*)
powinien pojawić się dalej. Odpowiedź powstaje token po tokenie, a wybór
jest częściowo losowy. Nie ma tu reguł pisanych ręcznie jak w ELIZIE.

**Zapytanie do API** to **lista wiadomości** z rolami:

| Rola | Kto mówi |
|---|---|
| `system` | instrukcja dla modelu — kim ma być, jak odpowiadać |
| `user` | użytkownik |
| `assistant` | wcześniejsze odpowiedzi modelu |

W odpowiedzi dostajemy **jedną** nową wiadomość modelu. Zapamiętaj ten
schemat — w Zadaniu 6 okaże się ważniejszy, niż wygląda.

**Klucz** podajesz przez `getpass` albo Colab *Secrets* (ikona klucza po lewej,
nazwa `GEMINI_API_KEY`). **Nigdy nie wklejaj klucza wprost do kodu.**

In [24]:
#!pip install openai        # odkomentuj, jeśli biblioteki nie ma (w Colabie jest)
import os
import time
from getpass import getpass
from openai import OpenAI

try:                                    # Colab: klucz z Secrets
    from google.colab import userdata
    API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:                       # lokalnie: zmienna środowiskowa albo getpass
    API_KEY = os.environ.get("GEMINI_API_KEY") or getpass("Wklej klucz Gemini API: ")

client = OpenAI(
    api_key=API_KEY,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

MODEL = "gemini-3.5-flash-lite"   # jeśli nie działa: sprawdź listę w następnej komórce
REASONING = "minimal"             # ogranicza „myślenie" modelu; przy błędzie 400 ustaw None
print("Klient gotowy. Model:", MODEL)

Klient gotowy. Model: gemini-3.5-flash-lite


In [25]:
# Jakie modele Flash są dostępne dla Twojego klucza?
for m in client.models.list():
    if "flash" in m.id:
        print(m.id)

models/gemini-2.5-flash
models/gemini-2.5-flash-preview-tts
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-3.1-flash-tts-preview
models/gemini-3.8-flash-tts
models/gemini-3.8-flash-lite-tts
models/gemini-2.5-flash-native-audio-latest
models/gemini-2.5-flash-native-audio-preview-09-2025
models/gemini-2.5-flash-native-audio-preview-12-2025
models/gemini-3.1-flash-live-preview


In [26]:
def czat(messages, temperatura=None, max_tokens=400):
    params = {"model": MODEL, "messages": messages, "max_tokens": max_tokens}
    if REASONING:
        params["reasoning_effort"] = REASONING
    if temperatura is not None:
        params["temperature"] = temperatura
    for proba in range(3):
        try:
            r = client.chat.completions.create(**params)
            return (r.choices[0].message.content or "").strip()
        except Exception as e:
            if "429" in str(e) and proba < 2:
                print("  (limit zapytań — czekam 20 s)")
                time.sleep(20)
            else:
                raise

SYSTEM_TERAPEUTA = ("Jesteś terapeutą prowadzącym rozmowę po polsku. "
                    "Odpowiadaj krótko, jednym–dwoma zdaniami.")

def zapytaj_llm(tekst, system=SYSTEM_TERAPEUTA):
    # JEDNO pytanie, BEZ historii rozmowy
    return czat([{"role": "system", "content": system},
                 {"role": "user",   "content": tekst}])

print(zapytaj_llm("Czuję się dzisiaj samotny."))   # test połączenia

Przykro mi, że tak się czujesz. Co sprawia, że ta samotność jest dziś szczególnie trudna?


**Coś nie działa?**

| Objaw | Przyczyna |
|---|---|
| `401` / `API key not valid` | źle skopiowany klucz — uruchom komórkę z kluczem jeszcze raz |
| `404` / `model not found` | nazwa modelu się zmieniła — wybierz z listy wyżej |
| `400` z `reasoning_effort` | ustaw `REASONING = None` i uruchom komórki ponownie |
| `429` | limit zapytań — odczekaj minutę; nie uruchamiaj pętli wielokrotnie |
| pusta odpowiedź `""` | model zużył limit tokenów na „myślenie" — zwiększ `max_tokens` |

## 6. Ta sama rozmowa, dwa silniki

### 🔧 Zadanie 4
1. Uzupełnij pętlę tak, żeby każde zdanie z `TESTY` trafiło do **obu** systemów.
2. Zapisz **3 konkretne różnice**. Nie oceniaj, który jest „lepszy" —
   opisz, **co** każdy z nich robi z Twoim zdaniem.

*Na co patrzeć:* czy odpowiedź powtarza Twoje słowa, czy dodaje coś nowego?
Czy jest gramatyczna? Czy odnosi się do sensu, czy do słowa kluczowego?

In [27]:
for zdanie in TESTY:
    odp_eliza = odpowiedz(zdanie)
    odp_llm = zapytaj_llm(zdanie)                     # <- uzupełnij (Zadanie 4)
    print(f"> {zdanie}\n  ELIZA: {odp_eliza}\n  LLM:   {odp_llm}\n")

> Potrzebuję, żeby moja rodzina mnie słuchała.
  ELIZA: Dlaczego potrzebujesz żeby twoja rodzina ciebie słuchała?
  LLM:   To całkowicie naturalna potrzeba, że chcesz być usłyszany i zrozumiany przez najbliższych. Co dokładnie czujesz, kiedy czujesz, że Cię ignorują?

> Czuję się dzisiaj samotny.
  ELIZA: Czy często tak się czujesz?
  LLM:   Przykro mi, że tak się czujesz. Co sprawia, że ta samotność jest dziś szczególnie trudna?

> Mój ojciec ciągle mnie krytykuje.
  ELIZA: Opowiedz mi więcej o swojej rodzinie.
  LLM:   To musi być bardzo bolesne, kiedy bliska osoba zamiast wsparcia daje Ci poczucie, że robisz wszystko źle. Co dokładnie zazwyczaj u niego wywołuje taką reakcję?

> Nie wiem, o co ci chodzi.
  ELIZA: Rozumiem. Mów dalej.
  LLM:   Rozumiem, że to może być na ten moment niejasne. O czym chciałbyś dzisiaj porozmawiać?

> Wszyscy mnie ignorują.
  ELIZA: Rozumiem. Mów dalej.
  LLM:   To musi być dla Ciebie bardzo bolesne i samotne doświadczenie. Chcesz o tym opowiedzieć więce

**Różnice (Zadanie 4):**

1. Eliza wyszukuje słowa kluczowego i na jego podstawie dopasowuje odpowiedź, LLM patrzy na całe zdanie, a nie jeden wyraz
2. Odpowiedzi Elizy są proste i krótkie, LLM rozbudowuje zdania
3. LLM zadaje dużo więcej pytań niż Eliza

## 7. Łamiące zdania — druga runda

### 🔧 Zadanie 5
1. Uruchom porównanie na **Twoich** zdaniach z Zadania 3 (lista `przyklady_lamiace`
   jest już zdefiniowana wyżej).
2. **Zanim uruchomisz** test pamięci, zapisz swoje przewidywanie:
   czy LLM odpowie poprawnie na drugie pytanie?
3. Uzupełnij tabelę.

In [28]:
for zdanie in przyklady_lamiace:
    print(f"> {zdanie}\n  ELIZA: {odpowiedz(zdanie)}\n  LLM:   {zapytaj_llm(zdanie)}\n")

> Pożyczysz mi 100 zł?
  ELIZA: Jak się z tym czujesz?
  LLM:   Przykro mi, ale nie mogę pożyczyć ci pieniędzy, ponieważ jestem twoim terapeutą. Porozmawiajmy o tym, co sprawia, że potrzebujesz teraz wsparcia finansowego.

> Dlaczego nie chciałeś pożyczyć mi swojego kabla?
  ELIZA: Rozumiem. Mów dalej.
  LLM:   Przepraszam, jeśli poczułeś się urażony, ale po prostu nie pożyczam swoich rzeczy osobistych. Czy chciałbyś poszukać innego sposobu na rozwiązania tego problemu?

> Czy zgodnie z tym co mówiłeś mi wcześniej, skończyłeś już zbierać pomidory ze swojego ogródka?
  ELIZA: Co masz na myśli?
  LLM:   Pomidory to wdzięczny temat, ale skupmy się dzisiaj na Tobie i tym, co Ty czujesz. Co sprawia, że o to pytasz?

> Za ile chciałbyś sprzedać mi swój dom?
  ELIZA: Jak się z tym czujesz?
  LLM:   Nie jestem właścicielem żadnego domu, jestem tu jako twój terapeuta. Porozmawiajmy o tym, co skłoniło Cię do zadania tego pytania.



**Moje przewidywanie (przed uruchomieniem):Myślę że nie, ponieważ nie przechowuje informacji o rozmowach z przeszłości, a więc zacznie wymyślać.

In [30]:
# Test pamięci: dwa OSOBNE wywołania, jedno po drugim
print("LLM:", zapytaj_llm("Mój brat ma na imię Tomek i ciągle mnie krytykuje."))
print("LLM:", zapytaj_llm("Jak ma na imię mój brat?"))

LLM: To musi być dla ciebie bardzo bolesne, kiedy słyszysz takie słowa od bliskiej osoby. Jak zazwyczaj reagujesz na te jego uwagi?
LLM: Jako terapeuta nie znam szczegółów z Twojego życia prywatnego. Chcesz o nim opowiedzieć?


**Tabela (Zadanie 5):**

| Typ pęknięcia | ELIZA | LLM | Dlaczego? |
|---|---|---|---|
| negacja | ❌ | radzi sobie lepiej od Elizy | interpretuje zdania zamiast pojedynczych słów kluczowych |
| ironia | ❌ | Lepiej od Elizy ale ciągle łatwo go tym złamać | bazuje na zdaniach zamiast jednym słowie ale ciągle nie rozumie kontekstu  |
| polska gramatyka / odbicie zaimków | ❌ | Tworzy poprawną choć często nienaturalnie rozbudowaną odpowiedź | Generuje pełne zdania zamiast bazowania na tablicy ze słowami kluczowymi |
| pamięć poprzednich zdań | ❌ | Nie posiada pamięci z poprzednich rozmów choć ma tu większy potencjał niż Eliza, gdyby miał dostęp do historii rozmów  | Eliza bazuje na słowach kluczowych i historia rozmów bardzo mocno rozbudowywałaby kod, a jednocześnie generowała niezbyt imponujący efekt w porównaniu do LLM |

## 8. Pamięć to lista wiadomości

Model **nie przechowuje** rozmowy między wywołaniami. To, że czatboty
„pamiętają", wynika z tego, że program **odsyła modelowi całą dotychczasową
rozmowę** przy każdym nowym pytaniu.

### 🔧 Zadanie 6
a) i b) Uzupełnij `rozmowa_z_pamiecia`: po każdej turze dopisuj do `historia`
wypowiedź użytkownika (rola `user`) i odpowiedź modelu (rola `assistant`).
c) Uruchom test i komórkę mierzącą historię. Odpowiedz: **gdzie jest pamięć —
w modelu czy w Twoim kodzie?** Co rośnie z każdą turą i jakie to ma konsekwencje?

In [33]:
def rozmowa_z_pamiecia(wejscia, system=SYSTEM_TERAPEUTA):
    historia = [{"role": "system", "content": system}]
    for zdanie in wejscia:
        # Zadanie 6a: dopisz wypowiedź użytkownika do historii
        historia.append({"role": "user", "content": zdanie})
        odp = czat(historia)
        # Zadanie 6b: dopisz odpowiedź modelu do historii
        historia.append({"role": "assistant", "content": odp})
        print(f"Ty:  {zdanie}\nLLM: {odp}\n")
    return historia

In [35]:
historia = rozmowa_z_pamiecia([
    "Mój brat ma na imię Tomek i ciągle mnie krytykuje.",
    "Wczoraj znowu się pokłóciliśmy.",
    "Jak ma na imię mój brat?",
])

Ty:  Mój brat ma na imię Tomek i ciągle mnie krytykuje.
LLM: Słyszę, że te uwagi bardzo cię bolą. Co dokładnie zazwyczaj mówi Tomek?

Ty:  Wczoraj znowu się pokłóciliśmy.
LLM: Współczuję ci, takie konflikty potrafią bardzo obciążać. O co dokładnie poszło tym razem?

Ty:  Jak ma na imię mój brat?
LLM: Twój brat ma na imię Tomek. Co najbardziej denerwuje cię w jego zachowaniu?



In [36]:
# Co faktycznie wysłaliśmy modelowi przy ostatnim pytaniu?
for w in historia:
    print(f"[{w['role']:9}] {w['content'][:70]}")
print("\nLiczba wiadomości:", len(historia))
print("Liczba znaków wysłanych przy ostatnim zapytaniu:",
      sum(len(w["content"]) for w in historia[:-1]))

[system   ] Jesteś terapeutą prowadzącym rozmowę po polsku. Odpowiadaj krótko, jed
[user     ] Mój brat ma na imię Tomek i ciągle mnie krytykuje.
[assistant] Słyszę, że te uwagi bardzo cię bolą. Co dokładnie zazwyczaj mówi Tomek
[user     ] Wczoraj znowu się pokłóciliśmy.
[assistant] Współczuję ci, takie konflikty potrafią bardzo obciążać. O co dokładni
[user     ] Jak ma na imię mój brat?
[assistant] Twój brat ma na imię Tomek. Co najbardziej denerwuje cię w jego zachow

Liczba wiadomości: 7
Liczba znaków wysłanych przy ostatnim zapytaniu: 354


**Odpowiedź (Zadanie 6c):Pamięć jest w kodzie, który wysyłamy modelowi. Z każdą turą rośnie coraz bardziej nasza wiadomość którą wysyłamy, wraz ze wzrostem historii naszych poprzednich wiadomości. Konsekwencją jest zbliżanie się do limitu wygenerowanych przez nas zapytań.
### ★ Zadanie 6d (dla chętnych) — fałszywa pamięć
Skoro pamięć to lista, którą **Ty** budujesz — możesz ją sfałszować. Dopisz
wiadomość `assistant`, której model nigdy nie powiedział, i sprawdź, czy
model „przyzna się" do tej wypowiedzi.

In [ ]:
falszywa = [
    {"role": "system", "content": SYSTEM_TERAPEUTA},
    {"role": "user", "content": "Mój brat ma na imię Tomek."},
    # TODO (Zadanie 6d): dopisz wiadomość 'assistant', której model nigdy nie wypowiedział
    {"role": "user", "content": "Co mi przed chwilą poradziłeś?"},
]
print(czat(falszywa))

## 9. Rola to tekst

W ELIZIE rola terapeuty była **zakodowana w regułach**. W LLM rolę ustala
jeden akapit tekstu w wiadomości `system`.

### 🔧 Zadanie 7
a) Dopisz do `ROLE` **jedną własną rolę** i uruchom to samo zdanie przez
wszystkie role. Czy LLM udający ELIZĘ łamie się tam, gdzie prawdziwa ELIZA?

b) Uruchom to samo pytanie trzy razy przy dwóch temperaturach. Porównaj
losowość LLM z losowością ELIZY (`random.choice`) — skąd bierze się w każdym
z systemów?

In [37]:
ROLE = {
    "terapeuta": SYSTEM_TERAPEUTA,
    "eliza": ("Jesteś programem ELIZA z 1966 roku. Nic nie wiesz o świecie. "
              "Odpowiadaj wyłącznie krótkim pytaniem, które powtarza słowa rozmówcy, "
              "np. 'Dlaczego mówisz, że ...?'. Nigdy nie udzielaj rad ani informacji."),
    "przewodnik": ("Jesteś przewodnikiem po Łodzi. Odpowiadaj krótko, po polsku, "
                   "zawsze proponując konkretne miejsce w mieście."),
    # TODO (Zadanie 7a): dopisz własną rolę
    "tłumacz": ("Jesteś tłumaczem języka japońskiego. Tłumaczysz z języka japońskiego na polski i odwrotnie "
                   "zawsze pytaj o szerszy kontekst, by wiedzieć do kogo i w jaki sposób tłumaczyć "),
}

zdanie = "Czuję się dzisiaj samotny."
for nazwa, system in ROLE.items():
    print(f"[{nazwa}] {zapytaj_llm(zdanie, system=system)}\n")

[terapeuta] Rozumiem, to bardzo trudny stan. Chcesz o tym opowiedzieć?

[eliza] Dlaczego mówisz, że czujesz się dzisiaj samotny?

[przewodnik] Rozumiem Cię. Najlepiej na samotność pomaga tłum ludzi i dobra energia – pójdź dzisiaj na **Ulicę Piotrkowską**. Spacerując najsłynniejszym deptakiem w Polsce, wśród kawiarnianych ogródków i muzyki ulicznych artystów, szybko poczujesz pulsujące życie miasta i łatwiej zapomnisz o melancholii.

[tłumacz] Rozumiem. Zanim przetłumaczę to zdanie na język japoński, **bardzo proszę o podanie szerszego kontekstu**:

1. **Do kogo to mówisz?** (do kogoś bliskiego, przyjaciół, czy może szukasz bardziej neutralnej formy?)
2. **Kim jesteś?** (w języku japońskim formy wypowiedzi mogą się nieznacznie różnić w zależności od płci osoby mówiącej).
3. **W jakiej sytuacji to piszesz?** (czy to wiadomość do ukochanej osoby, wpis w mediach społecznościowych, czy część dłuższego dialogu?)

Japonia ma bardzo bogaty system grzecznościowy i poziomów ekspresji, dlatego ko

In [ ]:
# Czy LLM-ELIZA łamie się tam, gdzie prawdziwa ELIZA?
for z in ["Nie potrzebuję niczego.", "Jak ma na imię mój brat?"]:
    print(f"> {z}\n  ELIZA:     {odpowiedz(z)}\n  LLM-ELIZA: {zapytaj_llm(z, system=ROLE['eliza'])}\n")

In [ ]:
# Zadanie 7b: to samo pytanie, 3 razy, dwie temperatury
pytanie = [{"role": "system", "content": SYSTEM_TERAPEUTA},
           {"role": "user", "content": "Nie mogę spać przed egzaminem."}]

for t in [0.0, 1.5]:
    print(f"--- temperatura {t} ---")
    for _ in range(3):
        print(" ", czat(pytanie, temperatura=t))

**Obserwacje (Zadanie 7):**

a) …

b) …

## 10. Refleksja

### 🔧 Zadanie 8 (3–5 zdań na punkt)
1. Dlaczego wybór roli **terapeuty rogeriańskiego** był dla ELIZY sprytny?
   Czy Twoja ELIZA działałaby równie przekonująco jako przewodnik turystyczny?
2. Które pęknięcia ELIZY LLM naprawił, a które zostały? Dlaczego pamięć okazała się
   „mechaniczna" także w LLM?
3. **Czy wrażenie rozumienia ma w obu systemach to samo źródło?** Uzasadnij na
   podstawie swoich wyników. Odwołaj się do obaw Weizenbauma.

**Twoje odpowiedzi (Zadanie 8):**

1. …
2. …
3. …

## Co oddajesz i jak oceniam

- **Kod (Zadania 1, 2, 4, 6a–b, 7a):** działa, wyniki są w notebooku.
- **Zadanie 3 i 5:** trafne przykłady i wyjaśnienie *dlaczego* (z odwołaniem do kodu);
  przewidywanie zapisane *przed* testem pamięci.
- **Zadania 6c, 7, 8:** samodzielne obserwacje i refleksja oparte na Twoich wynikach.
- **Zadanie 6d ★:** dodatkowe.

**Przed oddaniem** upewnij się, że klucz API nie znajduje się nigdzie w notebooku.